# Week 11 Lab — A model of the Tucson basin

**HWRS 564a · Fall 2026**

Last week's model had two boundaries and nothing in between, and you could
predict its answer on paper. This week you add a pumping well, and the head
field stops being something you could work out by hand.

The grid is the real basin: 15 km by 10 km at 250 m resolution, with land surface
elevations fitted to the 1,693 USGS wells you have been working with since
Week 5. The aquifer beneath it is confined — Week 12 takes the confining layer
off and shows you what that costs.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked  **`# YOUR TURN`**  have
something for you to write. Cells marked **`# CHECK`** verify your answer — if
they run without complaint, you're right.

> **Before you submit anything all semester:** *Kernel → Restart Kernel and Run
> All Cells*. A notebook that only works when run out of order is not finished.


## Learning objectives

By the end of this notebook you can:

1. Build a `DIS` package from a real land-surface array
2. Set constant-head boundaries that produce a gradient you intended
3. Add `WEL` and `RCH`, and get the signs right
4. Map heads with `PlotMapView`, and draw a cross-section
5. Read the water budget and say where the pumped water came from
6. Convert modelled heads to depth-to-water and compare with real measurements

---

## Part 1 — Setting up

The same four lines of boilerplate as last week. They go at the top of every
MODFLOW notebook you write.

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
DATA = ROOT / "data"
MF_EXE = ROOT / "modflow" / "mf2005"

assert MF_EXE.exists(), (
    f"MODFLOW binary not found at {MF_EXE}. Your codespace may not have "
    "finished building — run ./postbuild.sh from a terminal."
)

WS = ROOT / "_run" / "week11_basin"
WS.mkdir(parents=True, exist_ok=True)
print(f"workspace: {WS.relative_to(ROOT)}")

### The land surface

`data/tucson_grid_top.csv` is a 40 × 60 array of land-surface elevations in
metres, on our model grid.

It is a **second-order trend surface fitted to the 1,693 well elevations**, not
a DEM. Interpolating the wells directly gives 8 m of relief between adjacent
250 m cells, which is survey scatter rather than topography. The fit captures
basin form and leaves the mountain front in the residual. RMSE is about 36 m
over a 760 m spread — fine for teaching, not fine for anything else.

In [ ]:
land_surface = np.loadtxt(DATA / "tucson_grid_top.csv", delimiter=",")

NLAY, NROW, NCOL = 1, 40, 60
DELR = DELC = 250.0                # m

assert land_surface.shape == (NROW, NCOL), land_surface.shape
print(f"grid: {NROW} x {NCOL} cells of {DELR:.0f} m "
      f"= {NCOL * DELR / 1000:.1f} km x {NROW * DELC / 1000:.1f} km")
print(f"land surface: {land_surface.min():.0f} to {land_surface.max():.0f} m")
print(f"row 0 is the NORTH edge; column 0 is the WEST edge")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.2))
im = ax.imshow(land_surface, cmap="terrain",
               extent=[0, NCOL * DELR / 1000, 0, NROW * DELC / 1000])
cs = ax.contour(np.flipud(land_surface), levels=np.arange(650, 1000, 50),
                colors="k", linewidths=0.6,
                extent=[0, NCOL * DELR / 1000, 0, NROW * DELC / 1000])
ax.clabel(cs, fmt="%.0f", fontsize=8)
fig.colorbar(im, ax=ax, label="land surface elevation (m)")
ax.set_xlabel("distance east (km)")
ax.set_ylabel("distance north (km)")
ax.set_title("Model domain: the valley axis runs north-west")
plt.tight_layout()
plt.show()

Elevations rise to the south-east, toward the Santa Rita and Catalina foothills,
and fall toward the north-west along the valley axis. **So groundwater flows
north-west**, and our boundary conditions have to say so.

---

## Part 2 — The aquifer

We model the basin-fill aquifer as one confined layer between 400 m and 600 m
elevation — beneath the land surface, which is everywhere above 660 m.

In [ ]:
AQ_TOP, AQ_BOT = 600.0, 400.0      # m elevation

mf = flopy.modflow.Modflow("basin", model_ws=str(WS), exe_name=str(MF_EXE))

flopy.modflow.ModflowDis(
    mf, NLAY, NROW, NCOL,
    delr=DELR, delc=DELC,
    top=AQ_TOP, botm=AQ_BOT,
    nper=1, steady=True,
)
print(f"aquifer thickness: {AQ_TOP - AQ_BOT:.0f} m")
print(f"land surface sits {land_surface.min() - AQ_TOP:.0f} to "
      f"{land_surface.max() - AQ_TOP:.0f} m above the aquifer top")

> **`top` here is the top of the *aquifer*, not the land surface.** The two are
> different things and MODFLOW only knows about the first one. The land surface
> matters to us for computing depth to water, which is what a well measures —
> and we do that at the end.
>
> Because head will come out above 600 m everywhere, every cell is **confined**:
> the full 200 m is saturated and transmissivity does not depend on head. That
> is what makes this model solve on the first try.

In [ ]:
# BAS — heads fall from 700 m in the south-east to 620 m in the north-west
ibound = np.ones((NLAY, NROW, NCOL), dtype=int)
ibound[:, :, 0] = -1        # west edge:  downgradient, constant head
ibound[:, :, -1] = -1       # east edge:  upgradient, constant head

WEST_HEAD, EAST_HEAD = 620.0, 700.0

strt = np.full((NLAY, NROW, NCOL), 660.0)
strt[:, :, 0] = WEST_HEAD
strt[:, :, -1] = EAST_HEAD

flopy.modflow.ModflowBas(mf, ibound=ibound, strt=strt)
print(f"head drop across the domain: {EAST_HEAD - WEST_HEAD:.0f} m")

### YOUR TURN 1

Check that the boundary conditions imply a gradient a hydrogeologist would
recognise. Remember from Week 10 that **constant heads sit at cell centres**, so
the flow distance is one cell shorter than the domain.

- `flow_distance_m` — centre-to-centre distance between the two boundaries
- `regional_gradient` — the dimensionless gradient
- `plausible` — `True` if the gradient is between 0.001 and 0.02, which is the
  usual range for a basin-fill aquifer

In [ ]:
# YOUR TURN
flow_distance_m = ...
regional_gradient = ...
plausible = ...

In [ ]:
# CHECK
assert abs(flow_distance_m - 14750.0) < 1e-6, f"got {flow_distance_m}"
assert abs(regional_gradient - 0.005424) < 1e-5, f"got {regional_gradient}"
assert plausible is True or plausible == np.True_, "check the range"
print(f"flow distance     {flow_distance_m:8.0f} m")
print(f"regional gradient {regional_gradient:8.5f}")
print("That is 0.0054 — the same order as the gradient you computed from two")
print("real wells back in Week 2. Correct.")

---

## Part 3 — Properties and stresses

In [ ]:
HK = 12.0             # m/d, horizontal hydraulic conductivity

flopy.modflow.ModflowLpf(mf, hk=HK, laytyp=0, ipakcb=53)

transmissivity = HK * (AQ_TOP - AQ_BOT)
print(f"K = {HK} m/d over {AQ_TOP - AQ_BOT:.0f} m  ->  T = {transmissivity:,.0f} m2/d")

`laytyp=0` is **confined**: transmissivity is `hk * thickness` and does not
change during the solve. That keeps the equations linear.

### The wellfield

One cell, pumping 20,000 m³/d.

In [ ]:
WELL_ROW, WELL_COL = 20, 30
WELL_Q = -20000.0          # m3/d, NEGATIVE = extraction

flopy.modflow.ModflowWel(
    mf, stress_period_data={0: [[0, WELL_ROW, WELL_COL, WELL_Q]]}, ipakcb=53
)
print(f"well at layer 0, row {WELL_ROW}, column {WELL_COL}")
print(f"rate {WELL_Q:,.0f} m3/d = {-WELL_Q * 365 / 1233.48:,.0f} acre-ft/yr")

> **A 250 m cell cannot resolve an individual well.** A real well is 0.3 m
> across; this cell is 62,500 m². What we are actually representing is a
> **wellfield** — several wells within one cell — and the head MODFLOW reports
> for that cell is an average over 6 hectares, not the water level you would
> measure inside a well casing.
>
> That distinction matters when someone asks you to match an observed pumping
> water level. You cannot, at this grid resolution, and saying so is the correct
> answer.

### Recharge

In [ ]:
RECHARGE = 1.2e-4          # m/d over every cell

flopy.modflow.ModflowRch(mf, rech=RECHARGE, ipakcb=53)

domain_area = NROW * NCOL * DELR * DELC
print(f"recharge rate  {RECHARGE:.1e} m/d = {RECHARGE * 365 * 1000:.0f} mm/yr")
print(f"total recharge {RECHARGE * domain_area:,.0f} m3/d over {domain_area / 1e6:.0f} km2")

> **Recharge landing on a constant-head cell is discarded.** MODFLOW applies
> `RCH` only where `ibound > 0`. Our two boundary columns are `-1`, so 80 of the
> 2,400 cells contribute nothing — and the budget will report less recharge than
> `rate × domain area` implies. That is not a rounding error; it is 600 m³/d.

### YOUR TURN 2

Before running: can recharge alone support the well?

- `n_active_cells` — cells with `ibound > 0`, which is where recharge lands
- `total_recharge_m3d` — recharge rate times the area of *those* cells
- `deficit_m3d` — how much more the well takes than recharge supplies
- `pct_from_boundaries` — that deficit as a percentage of the pumping rate

Whatever the well takes beyond recharge has to come from the constant-head
boundaries — which in a real basin means from somewhere else's water.

In [ ]:
# YOUR TURN
n_active_cells = ...
total_recharge_m3d = ...
deficit_m3d = ...
pct_from_boundaries = ...

In [ ]:
# CHECK
assert n_active_cells == 2320, f"expected 40 x 58 = 2320, got {n_active_cells}"
assert abs(total_recharge_m3d - 17400.0) < 1.0, f"got {total_recharge_m3d}"
assert abs(deficit_m3d - 2600.0) < 1.0, f"got {deficit_m3d}"
assert abs(pct_from_boundaries - 13.0) < 0.5, f"got {pct_from_boundaries}"
print(f"active cells         {n_active_cells:9,d} of {NROW * NCOL:,d}")
print(f"recharge supplies    {total_recharge_m3d:9,.0f} m3/d")
print(f"the well takes       {-WELL_Q:9,.0f} m3/d")
print(f"deficit              {deficit_m3d:9,.0f} m3/d  ({pct_from_boundaries:.0f}% of pumping)")
print("\nCorrect. Part 5 checks this against MODFLOW's own budget.")

---

## Part 4 — Solve it and look at it

In [ ]:
flopy.modflow.ModflowPcg(mf)
flopy.modflow.ModflowOc(
    mf, stress_period_data={(0, 0): ["save head", "save budget", "print budget"]}
)

mf.write_input()
success, buff = mf.run_model(silent=True, report=True)
assert success, "MODFLOW did not converge:\n" + "\n".join(buff[-20:])

head = flopy.utils.HeadFile(str(WS / "basin.hds")).get_data()
print(f"converged. head {head.min():.1f} to {head.max():.1f} m")
assert (head[0] > AQ_TOP).all(), "some cells came out unconfined — check AQ_TOP"
print("every cell is still confined, as intended")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))

mv = flopy.plot.PlotMapView(model=mf, ax=ax)
band = mv.plot_array(head[0], cmap="Blues_r")
cs = mv.contour_array(head[0], levels=np.arange(620, 705, 5.0),
                      colors="k", linewidths=0.7)
ax.clabel(cs, fmt="%.0f", fontsize=8)
mv.plot_bc("WEL", color="#AB0520")

ax.set_xlabel("easting (m)")
ax.set_ylabel("northing (m)")
ax.set_title("Steady-state head, with the wellfield in red")
fig.colorbar(band, ax=ax, label="head (m)", shrink=0.9)
plt.tight_layout()
plt.show()

The contours bend toward the well — that bend *is* the cone of depression, and on
a regional gradient this steep it shows up as a distortion of the flow field
rather than a bullseye.

### YOUR TURN 3

Quantify the drawdown. The well is at row 20; row 5 is far enough north to be
essentially undisturbed at the same column.

- `head_at_well` — head in the well cell
- `head_undisturbed` — head at row 5, same column
- `drawdown_m` — the difference

In [ ]:
# YOUR TURN
head_at_well = ...
head_undisturbed = ...
drawdown_m = ...

In [ ]:
# CHECK
assert abs(head_at_well - 654.36) < 0.2, f"got {head_at_well}"
assert abs(drawdown_m - 5.45) < 0.2, f"got {drawdown_m}"
print(f"head at the well   {head_at_well:8.2f} m")
print(f"undisturbed        {head_undisturbed:8.2f} m")
print(f"drawdown           {drawdown_m:8.2f} m")
print(f"\nthat is {100 * drawdown_m / (AQ_TOP - AQ_BOT):.1f}% of the aquifer thickness")
print("Correct.")

In [ ]:
# A cross-section through the wellfield row, and a reference row
x = (np.arange(NCOL) + 0.5) * DELR

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(x, head[0, 5, :], ls="--", color="#0C234B", lw=2.0,
        label="row 5 — away from the well")
ax.plot(x, head[0, WELL_ROW, :], ls="-", color="#AB0520", lw=2.5,
        label=f"row {WELL_ROW} — through the wellfield")
ax.fill_between(x, head[0, WELL_ROW, :], head[0, 5, :],
                color="#81D3EB", alpha=0.35)
ax.axvline((WELL_COL + 0.5) * DELR, color="#AB0520", lw=0.8, alpha=0.5)
ax.set_xlabel("distance east (m)")
ax.set_ylabel("head (m)")
ax.set_title("The cone of depression, drawn against an undisturbed row")
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Solid versus dashed, not just red versus blue — a Week 8 habit that carries over.

---

## Part 5 — Where did the water come from?

The head map tells you what happened. The **budget** tells you why.

In [ ]:
def budget_terms(list_path):
    """Every budget term from a MODFLOW .list file, as {name: (in, out)} in m3/d."""
    lines = Path(list_path).read_text().splitlines()
    start = max(i for i, l in enumerate(lines) if "VOLUMETRIC BUDGET" in l)
    block = lines[start:start + 30]

    half = "in"
    terms = {}
    for line in block:
        s = line.strip()
        if s.startswith("OUT:"):
            half = "out"
        if "=" not in s or s.startswith(("TOTAL", "IN - OUT", "PERCENT")):
            continue
        name = s.split("=")[0].strip()
        value = float(s.split()[-1])
        entry = terms.setdefault(name, {"in": 0.0, "out": 0.0})
        entry[half] = value
    return terms


terms = budget_terms(WS / "basin.list")
for name, v in terms.items():
    print(f"  {name:16s}  in {v['in']:12,.1f}   out {v['out']:12,.1f}")

### YOUR TURN 4

Confirm the prediction you made in Part 2 against what MODFLOW actually did.

- `net_boundary_inflow` — constant-head inflow minus constant-head outflow
- `recharge_in` — the recharge term
- `well_out` — the wells term

Then check that the net boundary contribution matches your `deficit_m3d`.

In [ ]:
# YOUR TURN
net_boundary_inflow = ...
recharge_in = ...
well_out = ...

In [ ]:
# CHECK
assert abs(recharge_in - 17400.0) < 1.0, f"got {recharge_in}"
assert abs(well_out - 20000.0) < 1.0, f"got {well_out}"
assert abs(net_boundary_inflow - deficit_m3d) < 5.0, (
    f"net boundary inflow {net_boundary_inflow:,.0f} should match your "
    f"predicted deficit {deficit_m3d:,.0f}"
)
print(f"recharge in            {recharge_in:9,.0f} m3/d")
print(f"net boundary inflow    {net_boundary_inflow:9,.0f} m3/d")
print(f"                       {'-' * 20}")
print(f"total in               {recharge_in + net_boundary_inflow:9,.0f} m3/d")
print(f"well out               {well_out:9,.0f} m3/d")
print("\nCorrect — and you predicted that number before running the model.")

**This is the sentence a model exists to let you write:** *of the 20,000 m³/d
pumped, 17,400 comes from recharge and 2,600 is captured from flow that would
otherwise have left the basin to the north-west.*

You cannot get that from a head map, and you cannot get it from a monitoring
well. It comes from the budget, and it is the part of the answer that a water
manager actually needs.

---

## Part 6 — Does it look like the real basin?

MODFLOW gives head as an **elevation**. A well measures **depth below land
surface**. The difference is the land surface array we started with.

In [ ]:
depth_to_water = land_surface - head[0]

print(f"modelled depth to water: {depth_to_water.min():.0f} to "
      f"{depth_to_water.max():.0f} m")

wells = pd.read_csv(DATA / "tucson_basin_wells.csv", dtype={"site_no": str})
levels = pd.read_csv(DATA / "tucson_water_levels.csv",
                     dtype={"site_no": str}, parse_dates=["date"])
observed = levels.groupby("site_no")["depth_to_water_m"].mean()
print(f"observed depth to water: {observed.min():.0f} to {observed.max():.0f} m "
      f"across {len(observed)} wells")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))

im = axes[0].imshow(depth_to_water, cmap="viridis_r",
                    extent=[0, NCOL * DELR / 1000, 0, NROW * DELC / 1000])
fig.colorbar(im, ax=axes[0], label="depth to water (m)")
axes[0].set_xlabel("distance east (km)")
axes[0].set_ylabel("distance north (km)")
axes[0].set_title("Modelled depth to water")

axes[1].hist(depth_to_water.ravel(), bins=30, alpha=0.7,
             color="#AB0520", label="modelled (all cells)", density=True)
axes[1].hist(observed.dropna(), bins=20, alpha=0.6,
             color="#0C234B", label="observed (80 wells)", density=True)
axes[1].set_xlabel("depth to water (m)")
axes[1].set_ylabel("density")
axes[1].set_title("Modelled vs. observed")
axes[1].legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

### YOUR TURN 5

The two distributions overlap but do not match. Quantify the mismatch.

- `modelled_median` — median modelled depth to water, over all cells
- `observed_median` — median observed depth, across the 80 monitored wells
- `bias_m` — modelled minus observed

In [ ]:
# YOUR TURN
modelled_median = ...
observed_median = ...
bias_m = ...

In [ ]:
# CHECK
assert 100 < modelled_median < 220, f"got {modelled_median}"
assert 30 < observed_median < 120, f"got {observed_median}"
assert abs(bias_m - (modelled_median - observed_median)) < 1e-9
print(f"modelled median  {modelled_median:7.1f} m")
print(f"observed median  {observed_median:7.1f} m")
print(f"bias             {bias_m:+7.1f} m")
print("Correct.")

**The model is biased deep by well over a hundred metres, and that is the most
useful thing in this notebook.**

Three reasons, and none of them are bugs:

1. **The boundary heads were invented.** I picked 620 m and 700 m to give a
   plausible gradient, not by fitting anything. They set the whole head field.
2. **The monitored wells are not a random sample of the basin.** They cluster
   where people pump, which is the valley floor, which is where water is
   shallowest.
3. **Uniform `K` and uniform recharge.** The real basin has neither.

A model that reproduces a head field you specified is not evidence of anything.
Making it match observations means adjusting the things you guessed until it
does — **calibration** — and being honest that a calibrated model is one
consistent explanation rather than the truth.

That is what Project 3 is about.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## What's due

**HW 9 — MODFLOW structure diagram**, Wednesday 11/11 at 11:59pm.

## Next week

Discretization and boundary conditions, and what happens when you take the
confining layer off this model.

## Stuck?

- `success` is `False` — read the last lines of `buff` and the `.list` file. On
  this model it usually means a boundary head below the aquifer top.
- Heads of `-1e30` are **dry cells**. They cannot happen in this notebook
  (`laytyp=0`), and they are Week 12's problem.
- `PlotMapView` with nothing showing usually means you passed `head` (3D)
  instead of `head[0]` (2D).
- A budget where `WELLS` is zero means the sign was positive — MODFLOW read it
  as injection and put it in the `IN` column.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.